# Carpet WaveToy Thorns

NRPy can generate a small Carpet/ETLegacy wave-equation application as three Einstein Toolkit thorns. This notebook runs the packaged generator in a temporary workspace and inspects the generated thorn structure. Building and running inside the Einstein Toolkit is outside this generation-only notebook.

[Index](../index.ipynb) | Previous: [ETLegacy Core Infrastructure](etlegacy_core_infrastructure.ipynb) | Next: [Backend Choice Guide](backend_choice_guide.ipynb)

## Why This Matters

The Carpet WaveToy example shows the wave equation packaged as Einstein Toolkit thorns. It is a generation and inspection workflow unless a full Einstein Toolkit environment is available.

## What You Will See

- Three generated Einstein Toolkit thorns.
- Representative CCL metadata files.
- A generated source and parameter-file inventory.

## Table of Contents

1. [Generator discovery](#Generator-discovery)
2. [Generate the three thorns](#Generate-the-three-thorns)
3. [Confirm representative files](#Confirm-representative-files)
4. [Inspect thorn responsibilities](#Inspect-thorn-responsibilities)
5. [Generated run artifacts](#Generated-run-artifacts)

## Generator Discovery

Discovery uses `importlib.util.find_spec` so the generator module is located without importing it at notebook startup.

In [1]:
import importlib.util
import re
import subprocess
import sys
import tempfile
from pathlib import Path

import nrpy

GENERATOR_MODULE = "nrpy.examples.carpet_wavetoy_thorns"

print("nrpy import succeeded")
generator_spec = importlib.util.find_spec(GENERATOR_MODULE)
print("generator discovered:", generator_spec is not None)
if generator_spec is None:
    raise RuntimeError(f"cannot locate {GENERATOR_MODULE}")


def display_command(command):
    return " ".join("python" if str(part) == sys.executable else str(part) for part in command)


def compact_output(text, max_lines=18):
    text = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", text)
    workspace = globals().get("WORKSPACE")
    project_dir = globals().get("PROJECT_DIR")
    if workspace is not None:
        text = text.replace(str(workspace), "temporary workspace")
    if project_dir is not None:
        text = text.replace(str(project_dir), f"project/et_wavetoy")
    text = text.replace(sys.executable, "python")
    lines = [line.rstrip() for line in text.replace("\r", "\n").splitlines() if line.strip()]
    if len(lines) > max_lines:
        lines = ["..."] + lines[-max_lines:]
    return "\n".join(lines)

nrpy import succeeded
generator discovered: True


## Generate the Three Thorns

The command below creates `project/et_wavetoy/` under the temporary workspace. The generated arrangement contains:

- `IDWaveToyNRPy`, which sets initial data.
- `WaveToyNRPy`, which evolves the wave equation.
- `diagWaveToyNRPy`, which provides diagnostics.

In [2]:
workspace_manager = tempfile.TemporaryDirectory(prefix="nrpy-carpet-wavetoy-")
WORKSPACE = Path(workspace_manager.name).resolve()
PROJECT_DIR = WORKSPACE / "project" / "et_wavetoy"

command = [sys.executable, "-m", GENERATOR_MODULE]
result = subprocess.run(
    command,
    cwd=WORKSPACE,
    text=True,
    capture_output=True,
    timeout=300,
)

print("command:", display_command(command))
print("return code:", result.returncode)
if result.stdout:
    print("stdout:")
    print(compact_output(result.stdout))
if result.stderr:
    print("stderr:")
    print(compact_output(result.stderr))
if result.returncode != 0:
    raise RuntimeError("Carpet WaveToy generation failed")

print("project directory: project/et_wavetoy")

command: python -m nrpy.examples.carpet_wavetoy_thorns
return code: 0
stdout:
...
Checking temporary workspace/project/et_wavetoy/IDWaveToyNRPy/interface.ccl...[written]
Checking temporary workspace/project/et_wavetoy/IDWaveToyNRPy/param.ccl...[written]
Checking temporary workspace/project/et_wavetoy/diagWaveToyNRPy/schedule.ccl...[written]
Checking temporary workspace/project/et_wavetoy/diagWaveToyNRPy/interface.ccl...[written]
Checking temporary workspace/project/et_wavetoy/diagWaveToyNRPy/param.ccl...[written]
Checking temporary workspace/project/et_wavetoy/WaveToyNRPy/src/WaveToyNRPy_MoL_registration.c...[written]
Checking temporary workspace/project/et_wavetoy/WaveToyNRPy/src/WaveToyNRPy_Symmetry_registration_oldCartGrid3D.c...[written]
Checking temporary workspace/project/et_wavetoy/WaveToyNRPy/src/WaveToyNRPy_rhs_eval.c...[written]
Checking temporary workspace/project/et_wavetoy/WaveToyNRPy/src/WaveToyNRPy_specify_Driver_BoundaryConditions.c...[written]
Checking temporary worksp

## Confirm Representative Files

Each thorn should contain its Cactus metadata files and `src/make.code.defn`.

In [3]:
required_files = [
    "IDWaveToyNRPy/interface.ccl",
    "IDWaveToyNRPy/param.ccl",
    "IDWaveToyNRPy/schedule.ccl",
    "IDWaveToyNRPy/src/make.code.defn",
    "WaveToyNRPy/interface.ccl",
    "WaveToyNRPy/param.ccl",
    "WaveToyNRPy/schedule.ccl",
    "WaveToyNRPy/src/make.code.defn",
    "diagWaveToyNRPy/interface.ccl",
    "diagWaveToyNRPy/param.ccl",
    "diagWaveToyNRPy/schedule.ccl",
    "diagWaveToyNRPy/src/make.code.defn",
]

missing = []
for relative_path in required_files:
    path = PROJECT_DIR / relative_path
    exists = path.exists()
    print(f"{relative_path}: {exists}")
    if not exists:
        missing.append(relative_path)

if missing:
    raise RuntimeError("missing generated files: " + ", ".join(missing))

IDWaveToyNRPy/interface.ccl: True
IDWaveToyNRPy/param.ccl: True
IDWaveToyNRPy/schedule.ccl: True
IDWaveToyNRPy/src/make.code.defn: True
WaveToyNRPy/interface.ccl: True
WaveToyNRPy/param.ccl: True
WaveToyNRPy/schedule.ccl: True
WaveToyNRPy/src/make.code.defn: True
diagWaveToyNRPy/interface.ccl: True
diagWaveToyNRPy/param.ccl: True
diagWaveToyNRPy/schedule.ccl: True
diagWaveToyNRPy/src/make.code.defn: True


## Inspect Thorn Responsibilities

The three thorns share one generated arrangement but have different roles. `WaveToyNRPy` owns the evolved gridfunctions and Method-of-Lines registration; the initial-data and diagnostic thorns write or compare fields through that public interface.

In [4]:
for thorn in ["IDWaveToyNRPy", "WaveToyNRPy", "diagWaveToyNRPy"]:
    thorn_dir = PROJECT_DIR / thorn
    print(f"\n{thorn}")
    for metadata_file in ["interface.ccl", "param.ccl", "schedule.ccl", "src/make.code.defn"]:
        path = thorn_dir / metadata_file
        line_count = len(path.read_text(encoding="utf-8").splitlines())
        print(f"  {metadata_file}: {line_count} lines")
    sources = sorted(path.name for path in (thorn_dir / "src").glob("*.c"))
    print("  generated C files:")
    for name in sources:
        print("   ", name)


IDWaveToyNRPy
  interface.ccl: 26 lines
  param.ccl: 16 lines
  schedule.ccl: 22 lines
  src/make.code.defn: 5 lines
  generated C files:
    IDWaveToyNRPy_exact_solution_all_points.c

WaveToyNRPy
  interface.ccl: 60 lines
  param.ccl: 12 lines
  schedule.ccl: 89 lines
  src/make.code.defn: 12 lines
  generated C files:
    WaveToyNRPy_MoL_registration.c
    WaveToyNRPy_Symmetry_registration_oldCartGrid3D.c
    WaveToyNRPy_rhs_eval.c
    WaveToyNRPy_specify_Driver_BoundaryConditions.c
    WaveToyNRPy_specify_NewRad_BoundaryConditions_parameters.c
    WaveToyNRPy_specify_aux_BoundaryConditions.c
    WaveToyNRPy_specify_evol_BoundaryConditions.c
    WaveToyNRPy_zero_rhss.c

diagWaveToyNRPy
  interface.ccl: 26 lines
  param.ccl: 10 lines
  schedule.ccl: 22 lines
  src/make.code.defn: 5 lines
  generated C files:
    diagWaveToyNRPy_exact_solution_all_points.c


## Generated Run Artifacts

The generated project also includes source files and, when provided by the packaged example, parameter and thorn-list artifacts for an Einstein Toolkit workflow. This notebook stops after generation because compiling and running the thorns requires a separate Einstein Toolkit environment.

In [5]:
interesting_files = sorted(
    path.relative_to(PROJECT_DIR)
    for path in PROJECT_DIR.rglob("*")
    if path.is_file()
    and (
        path.suffix in {".ccl", ".c", ".par"}
        or path.name == "make.code.defn"
        or "thorn" in path.name.lower()
    )
)

for relative_path in interesting_files[:80]:
    print(relative_path)

print("file count shown:", min(len(interesting_files), 80))
print("total matching files:", len(interesting_files))

IDWaveToyNRPy/interface.ccl
IDWaveToyNRPy/param.ccl
IDWaveToyNRPy/schedule.ccl
IDWaveToyNRPy/src/IDWaveToyNRPy_exact_solution_all_points.c
IDWaveToyNRPy/src/make.code.defn
WaveToyNRPy/interface.ccl
WaveToyNRPy/param.ccl
WaveToyNRPy/schedule.ccl
WaveToyNRPy/src/WaveToyNRPy_MoL_registration.c
WaveToyNRPy/src/WaveToyNRPy_Symmetry_registration_oldCartGrid3D.c
WaveToyNRPy/src/WaveToyNRPy_rhs_eval.c
WaveToyNRPy/src/WaveToyNRPy_specify_Driver_BoundaryConditions.c
WaveToyNRPy/src/WaveToyNRPy_specify_NewRad_BoundaryConditions_parameters.c
WaveToyNRPy/src/WaveToyNRPy_specify_aux_BoundaryConditions.c
WaveToyNRPy/src/WaveToyNRPy_specify_evol_BoundaryConditions.c
WaveToyNRPy/src/WaveToyNRPy_zero_rhss.c
WaveToyNRPy/src/make.code.defn
diagWaveToyNRPy/interface.ccl
diagWaveToyNRPy/param.ccl
diagWaveToyNRPy/schedule.ccl
diagWaveToyNRPy/src/diagWaveToyNRPy_exact_solution_all_points.c
diagWaveToyNRPy/src/make.code.defn
file count shown: 22
total matching files: 22


Generation confirms the Carpet/ETLegacy layout: metadata files describe each thorn, `make.code.defn` lists its generated C sources, and the source files implement initial data, evolution, common ETLegacy registration helpers, and diagnostics.

## Where This Leads

- [ETLegacy Core Infrastructure](etlegacy_core_infrastructure.ipynb) reviews the prerequisite step.
- [Backend Choice Guide](backend_choice_guide.ipynb) continues the tutorial path.
- [Index](../index.ipynb) returns to the full tutorial spine.